In [1]:
import pandas as pd

data = pd.read_excel('/content/BlinkIT Grocery Data.xlsx')
data.head()

,Item Fat Content,Item Identifier,Item Type,Outlet Establishment Year,Outlet Identifier,Outlet Location Type,Outlet Size,Outlet Type,Item Visibility,Item Weight,Sales,Rating
0,Regular,FDX32,Fruits and Vegetables,2012,OUT049,Tier 1,Medium,Supermarket Type1,0.100014,15.10,145.4786,5.0
1,Low Fat,NCB42,Health and Hygiene,2022,OUT018,Tier 3,Medium,Supermarket Type2,0.008596,11.80,115.3492,5.0
2,Regular,FDR28,Frozen Foods,2016,OUT046,Tier 1,Small,Supermarket Type1,0.025896,13.85,165.0210,5.0
3,Regular,FDL50,Canned,2014,OUT013,Tier 3,High,Supermarket Type1,0.042278,12.15,126.5046,5.0
4,Low Fat,DRI25,Soft Drinks,2015,OUT045,Tier 2,Small,Supermarket Type1,0.033970,19.60,55.1614,5.0


### Step 1: Standardize Column Names
Convert column names to snake_case to ensure consistency and compatibility across tools like Python, SQL, and Power BI.

In [2]:
data.columns = data.columns.str.lower().str.replace(' ', '_')

### Step 2: Standardize Item Fat Content Values

EDA identified inconsistent labels in the `item_fat_content` column
(e.g., "LF", "low fat", "reg", "Regular", "Low Fat").

These values were standardized into two consistent categories:
- Low Fat  
- Regular

This ensures accurate aggregation and comparison during analysis.

In [3]:
fat_map = {
    'LF'      : 'Low Fat',
    'low fat' : 'Low Fat',
    'Low Fat' : 'Low Fat',
    'reg'     : 'Regular',
    'Regular' : 'Regular'
}

data['item_fat_content']=data['item_fat_content'].map(fat_map)

### Step 3: Handle Missing Values in Item Weight

Approximately 17% of values were missing in the `item_weight` column.
Missing values were imputed using the median weight within each item type to preserve category-specific characteristics.

In [4]:
# Calculate missing percentage before imputation
null_before = data['item_weight'].isnull().mean() * 100

In [5]:
# Group-based median imputation
data['item_weight'] = data.groupby('item_type')['item_weight'] \
    .transform(lambda x: x.fillna(x.median()))



In [6]:
data['item_weight'].isnull().mean() * 100

np.float64(0.0)

### Step 4: Handle Zero Values in Item Visibility
EDA identified zero values in the item_visibility column.
Since a product cannot have zero display visibility if it
is available for sale, these zeros were treated as
missing data entry errors.

Zero values were replaced with the mean visibility
within each **item_type** group — mean was chosen over
median as visibility values contain no significant
outliers and mean better represents the typical
display allocation for each product category.

In [7]:
import numpy as np
data['item_visibility']=data['item_visibility'].replace(0,np.nan)

data['item_visibility'] = data.groupby('item_type')['item_visibility'].transform(lambda x:x.fillna(x.mean()))


### Step 5: Feature Engineering — Outlet Age

A new column `outlet_age` was derived from
`outlet_establishment_year` using the formula:

>    outlet_age = 2024 - outlet_establishment_year

**Purpose:**  
`outlet_age` improves interpretability by converting establishment year into a meaningful metric representing store maturity.

This enables analysis of how outlet age relates to sales performance, allowing comparison between newer and more established stores.

Older outlets may benefit from customer loyalty and brand familiarity, while newer outlets may reflect recent expansion strategies.

In [8]:
data['outlet_age'] = 2024 - data['outlet_establishment_year']

### Step 6: Data Modeling — Star Schema

The cleaned flat dataset (8,523 rows × 12 columns) was
restructured into a star schema consisting of one fact
table and two dimension tables.

**Why star schema?**
Loading a flat table directly into Power BI can lead to
inefficient filtering, slower calculations, and reduced
clarity in relationships.

A star schema improves:
- Query performance  
- Data model clarity  
- Accuracy of filtering across visuals  

**Tables created:**

| Table       | Rows  | Columns | Purpose                              |
|-------------|-------|---------|--------------------------------------|
| dim_item    | 1,559 | 5       | Product attributes (Primary Key)     |
| dim_outlet  | 10    | 6       | Outlet attributes (Primary Key)      |
| fact_sales  | 8,523 | 4       | Sales transactions (Foreign Keys)    |

**Storage format (CSV):**
- Lightweight and faster to load in Power BI  
- No formatting overhead compared to Excel  
- Compatible with BI tools and databases  
- Plain text format suitable for version control  

In [13]:
# Item dimension
dim_item = data[[
    'item_identifier','item_type',
    'item_fat_content','item_weight',
    'item_visibility']].drop_duplicates(subset='item_identifier').reset_index(drop=True)

# Outlet dimension
dim_outlet = data[[
    'outlet_identifier','outlet_type',
    'outlet_size','outlet_location_type',
    'outlet_establishment_year',
    'outlet_age']].drop_duplicates(subset='outlet_identifier').reset_index(drop=True)

# Fact table
fact_sales = data[['item_identifier','outlet_identifier',
                   'sales','rating']].reset_index(drop=True)

In [14]:
# Verify shapes
print(f'dim_item   : {dim_item.shape}')
print(f'dim_outlet : {dim_outlet.shape}')
print(f'fact_sales : {fact_sales.shape}')

dim_item   : (1559, 5)
dim_outlet : (10, 6)
fact_sales : (8523, 4)


In [15]:
# Save Tables to csv
dim_item.to_csv('dim_item.csv',index=False)
dim_outlet.to_csv('dim_outlet.csv',index=False)
fact_sales.to_csv('fact_sales.csv',index=False)

In [17]:
# Validate FK integrity before saving
# Purpose: ensure no orphaned records exist in fact table
assert fact_sales['item_identifier'].isin(
    dim_item['item_identifier']).all(), "item FK broken!"

assert fact_sales['outlet_identifier'].isin(
    dim_outlet['outlet_identifier']).all(), "outlet FK broken!"

print("All validations passed ✓")

All validations passed ✓
